In [ ]:
import sys

# Install dependencies
if not 'pandas' in sys.modules:
    !pip install pandas
if not 'plotly' in sys.modules:
    !pip install plotly
if not 'kaleido' in sys.modules:
    !pip install -U kaleido
if not 'ipykernel' in sys.modules:
    !pip install ipykernel
# !pip install --upgrade nbformat


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import visualization_tools as vt
import toolbox as tb
import plotly.graph_objects as go
import os

sns.set_theme()

ORIENTED = False
# ORIENTED = True

In [ ]:
diagramDir = "diagrams/"

if not os.path.exists(diagramDir):
    os.makedirs(diagramDir)

In [ ]:
statsDir_implicit = "../results/implicit_convergence/stats/"

In [ ]:

data_list = []

data_list.append(tb.open_data_all(statsDir_implicit, added_param=[["dataset", "Implicit"], ["type", "double"]], recursive=False))
data_all = pd.concat(data_list, ignore_index=True)
data_all = tb.clean_data(data_all)

# Print methods
print(data_all["Method"].unique())

# Timings (mean) from nanosecond to millisecond
data_all["Timings (mean)"] = data_all["Timings (mean)"] / 1e6

data_all_methods = data_all.copy()

all_methods, methods = tb.methods(data_all, ORIENTED)

data_all = data_all[data_all["Method"].isin(all_methods)]

color_map_method = {}
for method in all_methods:
    color_map_method[method] = plt.cm.get_cmap("tab20")(all_methods.index(method))

color_map_to_value = {} 
color_map_to_value_soft = {} 
for method in color_map_method.keys() :
    color_map_to_value[method] = "rgba(" + str(int(color_map_method[method][0]*255)) + ", " + str(int(color_map_method[method][1]*255)) + ", " + str(int(color_map_method[method][2]*255)) + ", 1)"
    color_map_to_value_soft[method] = "rgba(" + str(int(color_map_method[method][0]*255)) + ", " + str(int(color_map_method[method][1]*255)) + ", " + str(int(color_map_method[method][2]*255)) + ", 0.2)"

In [ ]:
properties = {
    "Mean curvature (mean)": tb.mean_curvature_estimation,
    "K1 mean": tb.principal_curvature_estimation,
    "K2 mean": tb.principal_curvature_estimation,
    "Timings (mean)": all_methods,
    "D1 mean": tb.curvature_direction,
    "D2 mean": tb.curvature_direction,
    "normal mean": tb.normal_estimation,
    "iShape mean": tb.principal_curvature_estimation
}

In [ ]:
title = "Convergence"

data_implicits = data_all[data_all["dataset"] == "Implicit"]
data_implicits = data_implicits[~data_implicits["Method"].isin( ["3DQuadric", "NormCov3D","NormCov2D","Cov2D", "PC-MLS", "Mean", "PCA"] )]

vt.base_fonts['line_width'] = 1

cols = [
    ("Method", properties["Mean curvature (mean)"], "Mean curvature (mean)", "RMSE Mean curvature (log)", "log"),
    ("Method", properties["K1 mean"], "K1 mean", "RMSE Min curvature (log)", "log"),
    ("Method", properties["D1 mean"], "D1 mean", "RMSAE Min direction"),
    ("Method", properties["normal mean"], "normal mean", "RMSAE Normal"),
]
rows = [
    ( "Noise position", [0], "N", "N" ),
    ( "Noise position", [0.015], "N", "N" ),
]

to_legend = "Method"

y_labels=["No noise", ""]

base_fonts = {
    'title_font': 18,
    'axis_title_font': 13,
    'axis_tick_font': 10,
    'annotation_font': 12,
    'legend_font' : 12,
    'font_factor': 1.35,
    'font_family': 'Times New Roman',
}

subplot_data = vt.create_subplots(data_implicits, rows, cols, to_legend, color_map_to_value)

fig = vt.create_modular_subplots(subplot_data, title=title, fonts=base_fonts, graph_ratio=0.55, page_width=(718.75), show_titles=False, collapse_x=False, collapse_y=False)
fig.show()
# fig.write_html(diagramDir + title + ".html")
fig.write_image(diagramDir + title + ".png", width=fig.layout.width, height=fig.layout.height)
fig.write_image(diagramDir + title + ".pdf", width=fig.layout.width, height=fig.layout.height)
print("Size of the figure : ", fig.layout.width, "x", fig.layout.height)